<a href="https://colab.research.google.com/github/Pujitha-pitta/Flyrank-ML-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*



**Finding 1: Growing content is longer and younger**

The paper observed that pages with increasing impressions were longer on average than declining pages (about 3,180 versus 2,311 words) and younger (184 versus 230 days). The paper correctly describes this as an observational comparison rather than proof that increasing word count or reducing age causes growth.

**Where does the label come from?**

The “growing” and “declining” labels come from the change in impressions between the latest 30-day period and the previous 30-day period. Pages with more than 10% growth are labelled “up,” while pages with more than 10% decline are labelled “down.”

My methodology question is whether seasonality, changes in search demand, campaigns, indexing changes, or external events could also influence this label. A page may be labelled as declining even when its content quality has not changed.

**Does the validation design carry the claim?**

The large samples support a directional association within this portfolio. However, the analysis does not establish that longer content causes growth. I would also ask whether the pattern remains stable when pages are grouped by client, topic, content type, and publication period. Therefore, the safe conclusion is that longer and younger pages were associated with stronger momentum in this dataset.


**Finding 2: Refreshed mature pages showed stronger performance**

The paper observed that content older than 365 days and refreshed within 30 days had approximately 3.2 times the Health Score and 57 times the impressions of mature pages that had not been refreshed recently.

**Where does the label come from?**

The “refreshed” label comes from the number of days since the page was last updated. A page updated within the previous 30 days is treated as recently refreshed.

My methodology question is how pages were selected for refreshing. Editors may have prioritised pages that already had strong historical demand, commercial importance, or better-quality content. This could create selection bias.

**Does the validation design carry the claim?**

The comparison supports an observed relationship between recent updates and stronger performance, but it does not by itself prove that refreshing caused the improvement. A stronger validation design would compare refreshed and untouched pages with similar historical traffic, age, topic, intent, and competition. A time-based before-and-after comparison would also help.

The safe interpretation is that recently refreshed mature pages showed stronger measured performance in this portfolio and may be useful candidates for decision-support testing.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42

DATA_URL = (
    "https://raw.githubusercontent.com/Pujitha-pitta/"
    "Flyrank-ML-internship/main/data/raw/"
    "content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)

print("Dataset shape:", df.shape)
print("Number of clients:", df["client_id"].nunique())
df.head()

Dataset shape: (30000, 44)
Number of clients: 32


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7




The target is a proxy for refresh need:

- `1` when `trend_direction == "down"`
- `0` when `trend_direction` is `"stable"` or `"up"`

Rows marked `"new"` or `"flat"` are excluded because they do not provide a sufficiently comparable trend signal.

This is a proxy label, not a manually verified decision that a page truly needs refreshing.

In [2]:
model_df = df[df["trend_direction"].isin(["up", "stable", "down"])].copy()

model_df["needs_refresh_proxy"] = (
    model_df["trend_direction"] == "down"
).astype(int)

print(model_df["needs_refresh_proxy"].value_counts())
print(model_df["needs_refresh_proxy"].value_counts(normalize=True))

needs_refresh_proxy
1    16262
0    10350
Name: count, dtype: int64
needs_refresh_proxy
1    0.611078
0    0.388922
Name: proportion, dtype: float64




Feature selection

I removed the direct target columns and the impression fields used to construct the target.

The target is based on the comparison between `impressions_last_30d` and
`impressions_prev_30d`. Keeping those variables would allow the model to
approximately reconstruct the label rather than learn a useful decision-support
pattern.

Identifiers are used only for splitting and are not model features.

In [3]:
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
]

categorical_features = [
    "content_type",
    "main_intent",
    "competition_level",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
]

selected_features = numeric_features + categorical_features

missing_columns = [
    column for column in selected_features
    if column not in model_df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

X = model_df[selected_features].copy()
y = model_df["needs_refresh_proxy"].copy()
groups = model_df["client_id"].copy()

print("Features used:", len(selected_features))
print("Rows used:", len(X))

Features used: 28
Rows used: 26612


In [5]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore"),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ]
)

def build_model():
    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "model",
                LogisticRegression(
                    max_iter=2000,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )

In [6]:
def evaluate_model(model, X_test, y_test, split_name):
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    return {
        "Split": split_name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_test, predictions, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(y_test, probabilities),
    }

**Before: random row split**

The random split can place pages belonging to the same client in both the
training and test sets. This may make evaluation easier because pages from one
client can share content strategy, audience, analytics patterns, and publishing
processes.

I use this result only as the “before” comparison.

In [7]:
X_train_random, X_test_random, y_train_random, y_test_random = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=RANDOM_STATE,
    )
)

random_model = build_model()
random_model.fit(X_train_random, y_train_random)

random_results = evaluate_model(
    random_model,
    X_test_random,
    y_test_random,
    "Random row split",
)

random_results

{'Split': 'Random row split',
 'Accuracy': 0.6400526019162127,
 'Precision': 0.7341506129597198,
 'Recall': 0.6443283123270827,
 'F1': 0.6863130320890635,
 'ROC-AUC': np.float64(0.6780586036523699)}

**After: grouped split by client**

For the honest evaluation, all pages from a client are kept entirely in either
the training set or the test set.

This tests whether the model transfers to unseen clients instead of recognising
patterns from clients already represented in training.

In [8]:
group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

train_indices, test_indices = next(
    group_splitter.split(X, y, groups=groups)
)

X_train_group = X.iloc[train_indices]
X_test_group = X.iloc[test_indices]
y_train_group = y.iloc[train_indices]
y_test_group = y.iloc[test_indices]

train_groups = set(groups.iloc[train_indices])
test_groups = set(groups.iloc[test_indices])

assert train_groups.isdisjoint(test_groups)

print("Training rows:", len(X_train_group))
print("Test rows:", len(X_test_group))
print("Training clients:", len(train_groups))
print("Test clients:", len(test_groups))
print("Client overlap:", len(train_groups.intersection(test_groups)))

grouped_model = build_model()
grouped_model.fit(X_train_group, y_train_group)

grouped_results = evaluate_model(
    grouped_model,
    X_test_group,
    y_test_group,
    "Grouped by client",
)

grouped_results

Training rows: 21514
Test rows: 5098
Training clients: 24
Test clients: 7
Client overlap: 0


{'Split': 'Grouped by client',
 'Accuracy': 0.6014123185562966,
 'Precision': 0.7071082992184593,
 'Recall': 0.604133545310016,
 'F1': 0.6515775034293553,
 'ROC-AUC': np.float64(0.6342010212977955)}

In [9]:
comparison = pd.DataFrame(
    [random_results, grouped_results]
).set_index("Split")

comparison.round(3)

,Accuracy,Precision,Recall,F1,ROC-AUC
Split,,,,,
Random row split,0.640,0.734,0.644,0.686,0.678
Grouped by client,0.601,0.707,0.604,0.652,0.634


**Before/after interpretation**

Under the random row split, the model measured an F1 score of 0.686
and ROC-AUC of 0.678

Under the grouped client split, it measured an F1 score of 0.652
and ROC-AUC of 0.634.

The grouped result is the more honest estimate for performance on unseen
clients. The difference between the two evaluations shows that the validation
design affects the measured result.

These numbers are observed on this dataset and should be used as
decision-support evidence, not as proof that the model will perform equally for
all future clients.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

 **Leakage audit**

The target is derived from the direction of recent impression change. Therefore,
features that directly contain the trend label or the two impression windows
used to calculate it must not be used for training.

Identifiers are also excluded because they may encourage memorisation rather
than transferable learning.

In [10]:
leakage_audit = pd.DataFrame(
    [
        {
            "Column": "trend_direction",
            "Decision": "Remove",
            "Reason": "Direct source of the target label",
        },
        {
            "Column": "trend_pct",
            "Decision": "Remove",
            "Reason": "Direct numeric representation of the target trend",
        },
        {
            "Column": "impressions_last_30d",
            "Decision": "Remove",
            "Reason": "Used to calculate the target trend",
        },
        {
            "Column": "impressions_prev_30d",
            "Decision": "Remove",
            "Reason": "Used to calculate the target trend",
        },
        {
            "Column": "content_id",
            "Decision": "Remove",
            "Reason": "Identifier; no transferable predictive meaning",
        },
        {
            "Column": "client_id",
            "Decision": "Split only",
            "Reason": "Used for grouping, not as a model feature",
        },
        {
            "Column": "clicks_last_30d / clicks_prev_30d",
            "Decision": "Remove from final feature set",
            "Reason": "Closely aligned recent-period outcomes may proxy the target",
        },
        {
            "Column": "sessions_last_30d / sessions_prev_30d",
            "Decision": "Remove from final feature set",
            "Reason": "Closely aligned recent-period outcomes may proxy the target",
        },
    ]
)

leakage_audit

,Column,Decision,Reason
0,trend_direction,Remove,Direct source of the target label
1,trend_pct,Remove,Direct numeric representation of the target trend
2,impressions_last_30d,Remove,Used to calculate the target trend
3,impressions_prev_30d,Remove,Used to calculate the target trend
4,content_id,Remove,Identifier; no transferable predictive meaning
5,client_id,Split only,"Used for grouping, not as a model feature"
6,clicks_last_30d / clicks_prev_30d,Remove from final feature set,Closely aligned recent-period outcomes may pro...
7,sessions_last_30d / sessions_prev_30d,Remove from final feature set,Closely aligned recent-period outcomes may pro...


In [11]:
forbidden_features = {
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "impressions_prev_30d",
    "clicks_last_30d",
    "clicks_prev_30d",
    "sessions_last_30d",
    "sessions_prev_30d",
    "content_id",
    "client_id",
    "needs_refresh_proxy",
}

leaked_features = forbidden_features.intersection(selected_features)

print("Leaked features found:", leaked_features)
assert not leaked_features, (
    f"Leakage detected in final feature set: {leaked_features}"
)

Leaked features found: set()


**Leakage conclusion**

No direct target-derived fields are present in the final feature set.

This does not guarantee that every remaining feature is independent of the
target. Several 90-day performance variables may still be correlated with recent
decline. However, they do not directly encode the label formula and are treated
as historical decision-support signals.

In [12]:
grouped_predictions = grouped_model.predict(X_test_group)
grouped_probabilities = grouped_model.predict_proba(
    X_test_group
)[:, 1]

error_examples = model_df.iloc[test_indices][
    [
        "content_id",
        "client_id",
        "content_type",
        "main_intent",
        "content_age_days",
        "days_since_last_update",
        "impressions_90d",
        "clicks_90d",
        "avg_position",
        "trend_direction",
        "needs_refresh_proxy",
    ]
].copy()

error_examples["prediction"] = grouped_predictions
error_examples["refresh_probability"] = grouped_probabilities
error_examples["error_type"] = np.select(
    [
        (
            (error_examples["needs_refresh_proxy"] == 1)
            & (error_examples["prediction"] == 0)
        ),
        (
            (error_examples["needs_refresh_proxy"] == 0)
            & (error_examples["prediction"] == 1)
        ),
    ],
    [
        "False negative",
        "False positive",
    ],
    default="Correct",
)

# Remove identifiers before displaying public evidence.
public_error_examples = error_examples.drop(
    columns=["content_id", "client_id"]
)

false_negatives = (
    public_error_examples[
        public_error_examples["error_type"] == "False negative"
    ]
    .sort_values("refresh_probability")
    .head(5)
)

false_positives = (
    public_error_examples[
        public_error_examples["error_type"] == "False positive"
    ]
    .sort_values("refresh_probability", ascending=False)
    .head(5)
)

print("False negatives: declining pages missed by the model")
display(false_negatives)

print("False positives: non-declining pages flagged by the model")
display(false_positives)

False negatives: declining pages missed by the model


,content_type,main_intent,content_age_days,days_since_last_update,impressions_90d,clicks_90d,avg_position,trend_direction,needs_refresh_proxy,prediction,refresh_probability,error_type
21565,keyword article,informational,445,104,309192,2689,2.0,down,1,0,0.023992,False negative
17127,keyword article,informational,487,104,83603,886,3.4,down,1,0,0.032641,False negative
26413,keyword article,informational,421,14,64718,544,3.1,down,1,0,0.069237,False negative
12064,keyword article,informational,421,25,46560,491,6.5,down,1,0,0.082913,False negative
18853,keyword article,informational,537,104,48413,416,4.7,down,1,0,0.088600,False negative


False positives: non-declining pages flagged by the model


,content_type,main_intent,content_age_days,days_since_last_update,impressions_90d,clicks_90d,avg_position,trend_direction,needs_refresh_proxy,prediction,refresh_probability,error_type
4065,keyword article,informational,141,20,4,0,6.0,up,0,1,0.874337,False positive
11089,keyword article,informational,117,20,79,1,3.2,up,0,1,0.871880,False positive
7017,keyword article,informational,126,20,35,0,5.0,up,0,1,0.868883,False positive
10614,keyword article,informational,117,20,41,0,4.8,stable,0,1,0.866513,False positive
6378,keyword article,informational,131,20,24,0,7.2,up,0,1,0.865244,False positive


**Error analysis**

The false negatives are important because they represent declining pages that
the model did not flag. Missing these pages could delay a useful content review.

The false positives represent stable or growing pages that the model marked as
possible refresh candidates. These errors could cause unnecessary editorial
work.

The failures show that the model should not automatically decide which pages
must be refreshed. Its output is better used as a directional decision-support
signal that helps a human prioritise pages for review.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*



**Original bold claim **

“The Logistic Regression model can accurately identify pages that need a
content refresh.”

**Revised public-safe claim**

“In the grouped client evaluation, the Logistic Regression model measured
0.652 F1 and 0.634 ROC-AUC for separating pages with declining
impression trends from stable or growing pages.

This result is directional and is based on a proxy label derived from recent
impression change. The model may support content-review prioritisation, but it
does not prove that every flagged page requires a refresh or that refreshing it
will improve performance.”

**Why I changed the claim**

The original sentence went beyond the available evidence. The target measures
declining impressions, not a manually verified need for refreshing. The revised
claim states what was actually observed and positions the model as
decision-support rather than an automated decision-maker.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.